# GBRT Demonstrators + Core Math

This notebook is intentionally minimal and keeps only:
- The core GBRT update equations
- Three interactive demonstrators in learning order
  1) Weak learner explorer
  2) Iteration scrubber
  3) Hyperparameter lab

In [1]:
import warnings

import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.tree import plot_tree

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

HAS_IPYWIDGETS = True
try:
    import ipywidgets as widgets
    from ipywidgets import interactive_output
    from IPython.display import display
except Exception:
    HAS_IPYWIDGETS = False
    print("ipywidgets not available. Running in static mode.")

In [2]:
def fit_gbrt_toy(n_estimators=120, learning_rate=0.08, max_depth=2, min_samples_leaf=8, noise=0.15, sample_size=300):
    rng = np.random.default_rng(RANDOM_STATE)
    x = np.linspace(0, 1, sample_size)
    y_true = np.sin(6 * x) + 0.4 * np.cos(14 * x)
    y = y_true + rng.normal(0, noise, size=sample_size)
    X = x.reshape(-1, 1)

    model = GradientBoostingRegressor(
        n_estimators=int(n_estimators),
        learning_rate=float(learning_rate),
        max_depth=int(max_depth),
        min_samples_leaf=int(min_samples_leaf),
        random_state=RANDOM_STATE,
        loss="squared_error",
    )
    model.fit(X, y)

    staged = np.array(list(model.staged_predict(X)))
    train_rmse = [np.sqrt(mean_squared_error(y, sp)) for sp in staged]

    return x, y, y_true, model, staged, np.array(train_rmse)

## GBRT Core Math

For `GradientBoostingRegressor(loss='squared_error')`, the objective is:

$$
\min_F \sum_{i=1}^n \frac{1}{2}\left(y_i - F(x_i)\right)^2
$$

Pseudo-residual at round $m$:

$$
r_{im} = y_i - F_{m-1}(x_i)
$$

Additive model update:

$$
F_m(x) = F_{m-1}(x) + \nu h_m(x)
$$

where $h_m$ is the new tree fitted to pseudo-residuals and $\nu$ is the learning rate.

## Demonstrators

The sections below are intentionally limited to the three visual labs.

### 1) Weak Learner Explorer (Tree-by-tree)

This visualizes the individual tree used at each boosting round (using `GradientBoostingRegressor` so each weak tree is inspectable).

In [3]:
def tree_explorer(tree_index=1, sample_index=0, n_estimators=60, learning_rate=0.08, max_depth=2, min_samples_leaf=8, noise=0.15, sample_size=250):
    x, y, y_true, model, staged, _ = fit_gbrt_toy(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        noise=noise,
        sample_size=sample_size,
    )

    tree_index = int(np.clip(tree_index, 1, n_estimators))
    sample_index = int(np.clip(sample_index, 0, sample_size - 1))

    tree = model.estimators_[tree_index - 1, 0]
    sample_x = np.array([[x[sample_index]]])
    path_nodes = tree.decision_path(sample_x).indices

    fig, ax = plt.subplots(figsize=(18, 6))
    plot_tree(tree, ax=ax, feature_names=["x"], filled=True, rounded=True, fontsize=8)
    ax.set_title(f"Weak learner #{tree_index}")
    plt.show()

    node_text = " -> ".join([str(n) for n in path_nodes])
    pred_contrib = model.learning_rate * tree.predict(sample_x)[0]
    print(f"Sample index: {sample_index}, x={x[sample_index]:.4f}")
    print(f"Node path: {node_text}")
    print(f"Tree contribution ν·h_m(x) = {pred_contrib:.6f}")


if HAS_IPYWIDGETS:
    controls3 = {
        "tree_index": widgets.IntSlider(value=1, min=1, max=60, step=1, description="tree_index"),
        "sample_index": widgets.IntSlider(value=0, min=0, max=249, step=1, description="sample_index"),
        "n_estimators": widgets.IntSlider(value=60, min=20, max=150, step=10, description="n_estimators"),
        "learning_rate": widgets.FloatSlider(value=0.08, min=0.01, max=0.30, step=0.01, description="learning_rate"),
        "max_depth": widgets.IntSlider(value=2, min=1, max=4, step=1, description="max_depth"),
        "min_samples_leaf": widgets.IntSlider(value=8, min=2, max=40, step=1, description="min_samples_leaf"),
        "noise": widgets.FloatSlider(value=0.15, min=0.01, max=0.5, step=0.01, description="noise"),
        "sample_size": widgets.IntSlider(value=250, min=120, max=500, step=10, description="sample_size"),
    }

    def _sync_tree_sample(change):
        controls3["tree_index"].max = int(controls3["n_estimators"].value)
        controls3["sample_index"].max = int(controls3["sample_size"].value - 1)
        controls3["tree_index"].value = min(controls3["tree_index"].value, controls3["tree_index"].max)
        controls3["sample_index"].value = min(controls3["sample_index"].value, controls3["sample_index"].max)

    controls3["n_estimators"].observe(_sync_tree_sample, names="value")
    controls3["sample_size"].observe(_sync_tree_sample, names="value")

    out3 = interactive_output(tree_explorer, controls3)
    display(widgets.VBox([
        widgets.HBox([controls3["tree_index"], controls3["sample_index"], controls3["n_estimators"], controls3["learning_rate"]]),
        widgets.HBox([controls3["max_depth"], controls3["min_samples_leaf"], controls3["noise"], controls3["sample_size"]]),
        out3,
    ]))
else:
    tree_explorer()

### 2) Iteration Scrubber: What tree `m` is doing

At each boosting iteration, a new tree fits pseudo-residuals $r_{im}=y_i-F_{m-1}(x_i)$ and updates the model.

In [4]:
def iteration_scrubber(m=1, n_estimators=120, learning_rate=0.08, max_depth=2, min_samples_leaf=8, noise=0.15, sample_size=300):
    x, y, y_true, model, staged, _ = fit_gbrt_toy(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        noise=noise,
        sample_size=sample_size,
    )

    m = int(np.clip(m, 1, n_estimators))

    pred_prev = np.full_like(y, model.init_.constant_[0], dtype=float) if m == 1 else staged[m - 2]
    pred_now = staged[m - 1]
    residual_prev = y - pred_prev
    contrib = pred_now - pred_prev

    rmse_prev = np.sqrt(mean_squared_error(y, pred_prev))
    rmse_now = np.sqrt(mean_squared_error(y, pred_now))

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    axes[0, 0].scatter(x, y, s=10, alpha=0.5, label="observed")
    axes[0, 0].plot(x, pred_prev, color="tab:orange", lw=2, label=f"F_(m-1), m={m-1}")
    axes[0, 0].plot(x, pred_now, color="tab:red", lw=2, label=f"F_m, m={m}")
    axes[0, 0].set_title("Prediction before and after adding tree m")
    axes[0, 0].legend()

    axes[0, 1].scatter(x, residual_prev, s=10, alpha=0.6)
    axes[0, 1].axhline(0, color="k", linestyle="--")
    axes[0, 1].set_title("Pseudo-residual target before tree m")

    axes[1, 0].scatter(x, contrib, s=10, alpha=0.6, color="tab:green")
    axes[1, 0].axhline(0, color="k", linestyle="--")
    axes[1, 0].set_title("Contribution of tree m: ν·h_m(x)")

    axes[1, 1].bar(["RMSE before", "RMSE after"], [rmse_prev, rmse_now], color=["tab:orange", "tab:red"])
    axes[1, 1].set_title(f"Iteration improvement ΔRMSE = {rmse_prev - rmse_now:+.4f}")

    fig.suptitle(f"GBRT iteration scrubber | m={m}/{n_estimators}", fontsize=12)
    plt.tight_layout()
    plt.show()


if HAS_IPYWIDGETS:
    controls2 = {
        "m": widgets.IntSlider(value=1, min=1, max=120, step=1, description="m"),
        "n_estimators": widgets.IntSlider(value=120, min=20, max=350, step=10, description="n_estimators"),
        "learning_rate": widgets.FloatSlider(value=0.08, min=0.01, max=0.30, step=0.01, description="learning_rate"),
        "max_depth": widgets.IntSlider(value=2, min=1, max=5, step=1, description="max_depth"),
        "min_samples_leaf": widgets.IntSlider(value=8, min=2, max=40, step=1, description="min_samples_leaf"),
        "noise": widgets.FloatSlider(value=0.15, min=0.01, max=0.5, step=0.01, description="noise"),
        "sample_size": widgets.IntSlider(value=300, min=120, max=900, step=20, description="sample_size"),
    }

    def _link_max_estimators(change):
        controls2["m"].max = int(change["new"])
        if controls2["m"].value > controls2["m"].max:
            controls2["m"].value = controls2["m"].max

    controls2["n_estimators"].observe(_link_max_estimators, names="value")
    out2 = interactive_output(iteration_scrubber, controls2)
    display(widgets.VBox([
        widgets.HBox([controls2["m"], controls2["n_estimators"], controls2["learning_rate"]]),
        widgets.HBox([controls2["max_depth"], controls2["min_samples_leaf"], controls2["noise"], controls2["sample_size"]]),
        out2,
    ]))
else:
    iteration_scrubber()

### 3) Interactive Hyperparameter Lab (fit + residual + loss)
Use sliders to build intuition for GBRT behavior.

In [ ]:
def interactive_hparam_lab(n_estimators=120, learning_rate=0.08, max_depth=2, min_samples_leaf=8, noise=0.15, sample_size=300):
    x, y, y_true, model, staged, train_rmse = fit_gbrt_toy(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        noise=noise,
        sample_size=sample_size,
    )

    y_pred = staged[-1]
    residual = y - y_pred

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    axes[0, 0].scatter(x, y, s=14, alpha=0.5, label="observed")
    axes[0, 0].plot(x, y_true, "k--", lw=1.8, label="noise-free signal")
    axes[0, 0].plot(x, y_pred, color="tab:red", lw=2.2, label="GBRT fit")
    axes[0, 0].set_title("Fit on toy data")
    axes[0, 0].legend()

    axes[0, 1].scatter(y, y_pred, s=12, alpha=0.6)
    lims = [min(y.min(), y_pred.min()), max(y.max(), y_pred.max())]
    axes[0, 1].plot(lims, lims, "r--")
    axes[0, 1].set_title("Parity (pred vs actual)")
    axes[0, 1].set_xlabel("actual")
    axes[0, 1].set_ylabel("pred")

    axes[1, 0].scatter(x, residual, s=12, alpha=0.6)
    axes[1, 0].axhline(0, color="k", linestyle="--")
    axes[1, 0].set_title("Residuals")

    axes[1, 1].plot(np.arange(1, len(train_rmse) + 1), train_rmse, lw=2)
    axes[1, 1].set_title("Training RMSE across boosting rounds")
    axes[1, 1].set_xlabel("boosting round")
    axes[1, 1].set_ylabel("RMSE")

    fig.suptitle(
        f"GBRT knobs | trees={n_estimators}, lr={learning_rate}, depth={max_depth}, min_leaf={min_samples_leaf}",
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()


if HAS_IPYWIDGETS:
    controls = {
        "n_estimators": widgets.IntSlider(value=120, min=20, max=350, step=10, description="n_estimators"),
        "learning_rate": widgets.FloatSlider(value=0.08, min=0.01, max=0.30, step=0.01, readout_format=".2f", description="learning_rate"),
        "max_depth": widgets.IntSlider(value=2, min=1, max=5, step=1, description="max_depth"),
        "min_samples_leaf": widgets.IntSlider(value=8, min=2, max=40, step=1, description="min_samples_leaf"),
        "noise": widgets.FloatSlider(value=0.15, min=0.01, max=0.5, step=0.01, readout_format=".2f", description="noise"),
        "sample_size": widgets.IntSlider(value=300, min=120, max=900, step=20, description="sample_size"),
    }
    out = interactive_output(interactive_hparam_lab, controls)
    display(widgets.VBox([widgets.HBox(list(controls.values())[:3]), widgets.HBox(list(controls.values())[3:]), out]))
else:
    interactive_hparam_lab()